# Runway Perception — Colab Eğitim

Bu notebook modeli Colab'ın ücretsiz GPU'sunda eğitir. Lokal geliştirme (Intel Mac, GPU yok)
sadece pipeline doğrulaması içindir; asıl eğitim burada koşar.

**Akış:** GPU kontrol → repo → kurulum → LARD subset indir → split → eğit → eğriler → `best.pt` indir.

> Çalıştırma: `Runtime > Change runtime type > GPU` seçili olsun.

In [ ]:
# 1) GPU var mı?
!nvidia-smi

In [ ]:
# 2) Repo (kendi GitHub URL'inle değiştir)
REPO_URL = "https://github.com/<kullanici>/runway-perception.git"
!git clone $REPO_URL
%cd runway-perception

In [ ]:
# 3) Bağımlılıklar (Colab'da torch zaten kurulu; numpy<2 pinini burada zorlamaya gerek yok)
!pip install -q segmentation-models-pytorch albumentations datasets

In [ ]:
# 4) LARD V2 subset indir (~800 görüntü) + scenario bazlı split
!python -m src.data.download_lard --n 800 --out data/lard
!python -m src.data.split

In [ ]:
# 5) Eğitim (~25 epoch, config'ten). GPU'da amp otomatik açılır.
!python -m src.training.train --config configs/unet_r34.yaml

In [ ]:
# 6) Loss / IoU eğrileri
import json
import matplotlib.pyplot as plt

hist = json.load(open('outputs/history.json'))
ep = [h['epoch'] for h in hist]
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(ep, [h['train_loss'] for h in hist], label='train')
ax[0].plot(ep, [h['val_loss'] for h in hist], label='val')
ax[0].set_title('Loss'); ax[0].set_xlabel('epoch'); ax[0].legend()
ax[1].plot(ep, [h['val_iou'] for h in hist], label='val IoU')
ax[1].plot(ep, [h['val_dice'] for h in hist], label='val Dice')
ax[1].set_title('Val metrikleri'); ax[1].set_xlabel('epoch'); ax[1].legend()
plt.tight_layout(); plt.show()

In [ ]:
# 7) En iyi checkpoint'i lokale indir
from google.colab import files
files.download('outputs/best.pt')